# Chạy một phương pháp so sánh (Kaggle, GPU T4, bật Internet)

Đổi `METHOD` để chọn phương pháp (`methods/<METHOD>/` trong repo `baselines`), `RUNS` để chọn dataset.
Chỉ clone repo `baselines`: mã nguồn các phương pháp đã chép sẵn trong `third_party/`, không clone repo đồ án, không cài môi trường MonoGS.
Mỗi run cho **một file zip** trong `/kaggle/working/`. Tải zip về rồi chấm cục bộ:
`python baselines/common/score_zip.py --zip <zip> --dataset <dataset> --wandb`.

Mỗi phiên notebook chỉ chạy **một** `METHOD`: hai phương pháp có thể cài các gói trùng tên (rasterizer của SplaTAM và GS-ICP).

In [ ]:
METHOD = "splatam"  # một thư mục trong methods/ có adapter.py và setup_kaggle.sh

OFFICE0_LIT="/kaggle/input/replica-lunogs/office0/lit"
OFFICE0_EVEN="/kaggle/input/replica-lunogs/office0/even"
OFFICE0_AMB="/kaggle/input/datasets/nguyntrungan/office0"

ROOM0_LIT="/kaggle/input/datasets/nguyntrungan/replica-lunogs/room0/lit"
ROOM0_EVEN="/kaggle/input/datasets/nguyntrungan/replica-lunogs/room0/even"

RUNS = [OFFICE0_LIT, OFFICE0_EVEN]  # chạy tuần tự, theo thứ tự này
CKPT_EVERY = 100  # frame giữa hai checkpoint (0 = tắt), để chạy tiếp khi phiên bị ngắt
TIMEOUT_H = 11    # giới hạn mỗi run; phiên Kaggle tối đa 12 giờ cho TẤT CẢ các run
NO_MAP = False    # True: không đưa bản đồ cuối vào zip (zip nhỏ hơn, không render lại được)
OUT = "/kaggle/working"
CODE = "/tmp/code"  # mã nguồn ngoài /kaggle/working: output chỉ giữ các zip

In [ ]:
%%time
import sys
!rm -rf {CODE}/baselines && mkdir -p {CODE} && git clone -q --depth 1 https://github.com/Trung4n/baselines {CODE}/baselines && git -C {CODE}/baselines log -1 --format='baselines %H %s'
sys.path.insert(0, f"{CODE}/baselines/common")
import kaggle_runner as K

K.setup(METHOD)  # methods/<METHOD>/setup_kaggle.sh: cài phụ thuộc bằng uv, build extension cho GPU này

Log của mỗi run ở `<OUT>/<METHOD>__<scene>_<variant>/run.log` và nằm trong zip. Các run chạy **tuần tự**: FPS phải đo khi máy không bị chia sẻ.

In [ ]:
zips = [K.run(METHOD, d, OUT, CKPT_EVERY, TIMEOUT_H, no_map=NO_MAP) for d in RUNS]
zips

**Chạy tiếp sau khi phiên bị ngắt:** thêm output của phiên trước làm input, chép thư mục `<METHOD>__<scene>_<variant>` của run dở về `/kaggle/working`, rồi chạy ô dưới.
Frame của checkpoint được xử lý lại một lần; `timing.json` cộng các phiên và ghi rõ là đã resume.

In [ ]:
# !cp -r /kaggle/input/<output phiên trước>/splatam__office0_lit {OUT}/
# K.run(METHOD, OFFICE0_LIT, OUT, CKPT_EVERY, TIMEOUT_H, resume=True, no_map=NO_MAP)

In [ ]:
# Kiểm mọi zip của METHOD rồi xóa thư mục làm việc của run đã đóng gói: output chỉ còn các zip để tải về.
K.collect(METHOD, OUT)